# 02 · Agents and Tasks

> **Kernel:** `Phase 5 · CrewAI` · **Needs:** Ollama.

In CrewAI you don't write prompts directly: you describe **agents** and
**tasks**, and CrewAI writes the prompts. That's convenient, but only if you
know *exactly* what it writes. This notebook opens the box: you'll see the real
prompt, then learn to design personas and tasks, choose what context each task
receives, and read every part of the output.

**You will learn**
- The **actual prompt** CrewAI builds from an agent and a task
- Designing `role`, `goal`, `backstory`, and why the persona changes the output
- Designing tasks: `description`, `expected_output`, input templates
- Controlling which results a task sees with `context`
- Reading `CrewOutput` and `TaskOutput`, and saving results to files
- The agent settings that matter: `max_iter`, `max_execution_time`, delegation

**Prerequisites:** notebook 01.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')                       # hide noisy library warnings in the notebook
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 03_CREWAI folder
sys.path.insert(0, str(SECTION))
from common import get_llm, quiet_console              # import common BEFORE crewai (telemetry off)
quiet_console()                                         # hide CrewAI's decorative console panels

import asyncio
from crewai import Agent, Task, Crew, Process
from crewai.events import BaseEventListener
from crewai.events.types.llm_events import LLMCallStartedEvent

llm = get_llm()

## 1 · Why: the prompt is generated for you

When a task runs, CrewAI combines the agent's persona and the task into chat
messages for the model. To design agents well you need to see those messages.
CrewAI emits an **event** before every model call (notebook 10 covers events),
and that event carries the messages. A tiny listener can capture them:

In [ ]:
captured = []

class PromptSpy(BaseEventListener):
    # Records the messages of every model call CrewAI makes.
    def setup_listeners(self, bus):
        @bus.on(LLMCallStartedEvent)
        def on_llm_call(source, event):
            captured.append(event.messages)

spy = PromptSpy()        # listeners register themselves when created


explainer = Agent(role='Technical explainer', goal='Explain ideas simply',
                  backstory='A patient teacher who uses everyday analogies.', llm=llm)
task = Task(description='Explain what an API is.', expected_output='Two sentences.', agent=explainer)

captured.clear()
await Crew(agents=[explainer], tasks=[task]).kickoff_async()
await asyncio.sleep(0.3)                          # events are delivered asynchronously

for message in captured[0]:
    print(f"── {message['role'].upper()} ──\n{message['content']}\n")

That's the whole mechanism:

```
 SYSTEM:  You are {role}. {backstory}
          Your personal goal is: {goal}
 USER:    Current Task: {task.description}
          This is the expected criteria for your final answer: {task.expected_output}
          you MUST return the actual complete content as the final answer, not a summary.
          (+ context from earlier tasks, + tool instructions when the agent has tools)
```

So **role, backstory and goal are the system prompt**, and **description and
expected output are the user prompt**. Every word you put in them is prompt text.

## 2 · Designing the persona

The persona isn't decoration: it steers every answer the agent gives. Same task,
two backstories:

In [ ]:
TASK = 'Describe the benefit of a sidewalk delivery robot for a small grocery shop.'

personas = {
    'concise analyst': 'You are a concise analyst. You write short, factual sentences and never exaggerate.',
    'hype marketer':   'You are an enthusiastic marketer who loves big claims and exclamation marks.',
}
for name, backstory in personas.items():
    agent = Agent(role=name, goal='Describe product benefits', backstory=backstory, llm=llm)
    t = Task(description=TASK, expected_output='One short paragraph.', agent=agent)
    out = await Crew(agents=[agent], tasks=[t]).kickoff_async()
    print(f'{name} ({len(out.raw.split())} words):\n{out.raw}\n')

Same task, same model, very different output. Guidelines:

| Field | Write it as | Example |
|---|---|---|
| `role` | a job title the model can picture | "Senior support engineer" |
| `goal` | what *good* looks like for this agent | "Resolve the issue in the fewest steps" |
| `backstory` | behaviour and style: how it works, what it values, what it avoids | "You verify facts with tools and never guess." |

Keep personas **consistent with the tasks you give them**. A persona that loves
long, enthusiastic answers will fight a task that needs 20 words; notebook 07
measures exactly that fight.

## 3 · Designing tasks

`description` says what to do; `expected_output` says what "done" looks like.
The second matters more than it seems: it's the only definition of the output's
**shape**. Curly-brace placeholders are filled from `inputs` at kickoff, so one
crew can serve many requests:

In [ ]:
analyst = Agent(role='Market analyst', goal='Assess product ideas',
                backstory='You are concise and factual.', llm=llm)
assess = Task(
    description='Assess the idea "{idea}" for the {market} market.',
    expected_output='Exactly three bullet points: one opportunity, one risk, one next step.',
    agent=analyst,
)
crew = Crew(agents=[analyst], tasks=[assess])

for inputs in ({'idea': 'robot grocery delivery', 'market': 'Dutch'},
               {'idea': 'drone pharmacy delivery', 'market': 'Indian'}):
    out = await crew.kickoff_async(inputs=inputs)
    print(inputs, '\n', out.raw, '\n')

## 4 · Choosing what a task sees: `context`

In a sequential crew each task receives the **previous** task's output by
default. Often you want something else: a task that needs results from two
earlier tasks, or one that must *not* see an earlier draft. `context=[...]`
lists exactly which task outputs a task receives:

```
 facts ─────────────────────────┐
 opinion (a hot take) ── ✗      ├──► summary   (context=[facts]: sees facts, NOT the opinion)
```

In [ ]:
researcher = Agent(role='Researcher', goal='State facts', backstory='Precise and neutral.', llm=llm)
pundit = Agent(role='Pundit', goal='Give opinions', backstory='Opinionated and provocative.', llm=llm)
editor = Agent(role='Editor', goal='Write neutral summaries', backstory='Balanced and careful.', llm=llm)

facts = Task(description='List 2 facts about {topic}.', expected_output='2 bullet points', agent=researcher)
opinion = Task(description='Give a provocative one-line opinion about {topic}.', expected_output='one line', agent=pundit)
summary = Task(description='Write a neutral 2-sentence summary of {topic}.', expected_output='2 sentences',
               agent=editor, context=[facts])          # ← only the facts, not the opinion

out = await Crew(agents=[researcher, pundit, editor], tasks=[facts, opinion, summary]).kickoff_async(
    inputs={'topic': 'sidewalk delivery robots'})
for t in out.tasks_output:
    print(f'── {t.agent} ──\n{t.raw}\n')

The editor worked only from the facts, so the pundit's hot take couldn't leak
into the "neutral" summary. **Decide what each task sees**: it saves tokens and
prevents one agent's framing from contaminating another's work.

## 5 · Reading the output, and saving to files

A `CrewOutput` holds the final result and every task's `TaskOutput`:

| Field | Meaning |
|---|---|
| `result.raw` | the last task's output as text |
| `result.pydantic` / `result.json_dict` | the last task's structured output, if it had one (notebook 03) |
| `result.tasks_output` | a `TaskOutput` per task: `description`, `raw`, `agent`, `pydantic`, … |
| `result.token_usage` | tokens and model calls for the whole run |

A task can also write its result to a file with `output_file=`, handy for
reports and logs:

In [ ]:
report = Task(description='Write a 3-line status note about {topic}.', expected_output='3 lines',
              agent=editor, output_file='checkpoints/status_note.md', create_directory=True)
out = await Crew(agents=[editor], tasks=[report]).kickoff_async(inputs={'topic': 'the robot pilot'})

import pathlib
print('file contents:\n' + pathlib.Path('checkpoints/status_note.md').read_text())
print('usage:', out.token_usage.total_tokens, 'tokens,', out.token_usage.successful_requests, 'calls')

**A trap in `token_usage`.** That single one-line task reported many model
calls. In this version, usage is **accumulated on the LLM object**: every crew
whose agents share `llm` adds to the same counter, and `result.token_usage` reports
that running total, not this run's usage. Here's the effect, and the fix (a fresh
LLM object per run you want to measure, or subtracting the "before" value):

In [ ]:
def one_call_crew(model):
    a = Agent(role='Greeter', goal='Greet', backstory='Brief.', llm=model)
    return Crew(agents=[a], tasks=[Task(description='Say hi.', expected_output='hi', agent=a)])

shared = get_llm()
for i in range(1, 4):
    out = await one_call_crew(shared).kickoff_async()
    print(f'shared LLM, run {i}: reports {out.token_usage.successful_requests} model call(s)')
out = await one_call_crew(get_llm()).kickoff_async()
print(f'fresh LLM         : reports {out.token_usage.successful_requests} model call(s)')

Each run made **one** call, but the shared-LLM runs reported 1, 2, 3. For cost
tracking per request, give each run its own LLM object (cheap to create), or record
the counter before and after.

## 6 · Agent settings that matter

| Setting | Default | Meaning |
|---|---|---|
| `llm` | the default model | which model this agent uses (can differ per agent) |
| `tools` | none | what the agent can do (notebook 04) |
| `max_iter` | **25** | the most reasoning/tool steps per task before it must answer (the loop guard) |
| `max_execution_time` | none | a time limit (seconds) per task |
| `max_retry_limit` | 2 | retries when the agent's execution errors |
| `allow_delegation` | **False** | may it hand work to other agents in the crew? (notebook 05) |
| `verbose` | False | print the agent's steps (notebook 10) |
| `inject_date` | False | add today's date to the prompt (useful for time-sensitive tasks) |

`max_iter=25` is generous. For simple tasks, lower it: a stuck agent then fails
fast instead of burning 25 model calls.

**YAML projects.** The `crewai create crew <name>` CLI generates a project where
agents and tasks live in `agents.yaml` / `tasks.yaml` and are wired together with
the `@CrewBase`, `@agent`, `@task` and `@crew` decorators. It's the same objects
as here, just defined in config files, which non-engineers can edit.

## 7 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Vague `expected_output` | unpredictable shape and length | state the exact format ("3 bullets: …") |
| Persona that contradicts the task | the output drifts toward the persona | align backstory with the task's needs |
| Letting every task see everything | leaked framing, wasted tokens | explicit `context=[...]` |
| Leaving `max_iter` at 25 for simple tasks | slow, costly failures | a small limit for simple tasks |
| Forgetting that placeholders need inputs | a literal `{topic}` in the prompt | pass every placeholder in `inputs` |
| Reading `token_usage` from runs sharing one LLM object | inflated, cumulative numbers | a fresh LLM per measured run, or before/after deltas |

## 8 · Check yourself

<details><summary><b>1.</b> Which fields become the system prompt, and which the user prompt?</summary>

System: `role`, `backstory`, `goal` ("You are {role}. {backstory} Your personal goal is: {goal}"). User: the task's `description` and `expected_output`, plus any context and tool instructions.
</details>

<details><summary><b>2.</b> Why does the same task give different outputs for the analyst and the marketer?</summary>

The persona is part of the system prompt, so it steers tone, length and claims in every answer the agent writes.
</details>

<details><summary><b>3.</b> A summary task must not see an earlier draft. How?</summary>

Give it an explicit `context=[...]` listing only the tasks it should see. In a sequential crew it would otherwise receive the previous task's output.
</details>

<details><summary><b>4.</b> What does <code>max_iter</code> limit, and why lower it?</summary>

The number of reasoning/tool steps an agent may take on one task (default 25). Lowering it makes stuck or looping agents fail fast and cheaply.
</details>

## Takeaway

CrewAI turns **role/backstory/goal** into the system prompt and
**description/expected output** into the user prompt, so design them as the prompt
text they are. Make `expected_output` precise, keep personas consistent with the
tasks, choose each task's `context`, and set realistic limits.

Next → `03_llms_and_structured_output.ipynb`: models, and typed results.